# ENGG2112 Data Engineering

Objective: prepare reproducible data checks and a team handoff using the original **five APTOS 2019 grades from 0 to 4**.

This learning and working Notebook was written with AI assistance. Understand, check and revise each cell, and retain the conversation and your verification records. Do not describe generated work as independently completed research. Keep source data and image outputs local.

In VS Code, select the project's `.venv/Scripts/python.exe` as the kernel. Click the run button beside each code cell or press Shift+Enter. Start with Sections 1 to 4.

This full audit Notebook requires the original `data/train_images` folder. B can use `python -m examples.load_data_for_b` with the prepared-image handoff package instead.

## 1. Locate the project and data

`Path` represents a filesystem path; `ROOT` is the project root. Run this Notebook from the project root or the `notebooks` folder. The code does not depend on a particular computer's drive letter.

In [ ]:
from pathlib import Path
import sys
import json
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as DisplayImage

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'src/common.py').is_file():
    raise RuntimeError('Open the project folder in VS Code and run from the root or notebooks directory.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.common import CLASS_NAMES, read_labels, file_hash

LABELS = ROOT / 'data/train.csv'
IMAGES = ROOT / 'data/train_images'
print('Project:', ROOT)
print('CSV exists:', LABELS.is_file())
print('Image folder exists:', IMAGES.is_dir())
assert LABELS.is_file() and IMAGES.is_dir(), 'Place the official dataset in the data folder first.'


## 2. Read the label table

A CSV is a table, and `df` is that table loaded into Python. `id_code` matches the image filename; `diagnosis` is the supervised learning target y. We retain all five grades without mapping them to binary labels.

`read_labels` rejects missing, repeated or unsafe IDs and invalid grades rather than silently removing them.

In [ ]:
df = read_labels(LABELS)
display(df.head())
print('Rows, columns:', df.shape)
print('Columns:', list(df.columns))
display(df.isna().sum().rename('missing_count'))


## 3. Examine the five-class distribution

`value_counts()` counts the records in each grade. The proportion is the class count divided by the total number of records.

**Question:** What accuracy would a classifier achieve if it predicted the most frequent class for every image? This is the majority-class baseline; it does not demonstrate useful screening performance.

In [ ]:
counts = df['diagnosis'].value_counts().reindex(range(5), fill_value=0)
class_table = pd.DataFrame({
    'grade': range(5),
    'name': [CLASS_NAMES[i] for i in range(5)],
    'count': counts.values,
    'percent': (counts.values / len(df) * 100).round(2),
})
display(class_table)
print(f'Majority-class baseline: {counts.max() / len(df):.2%}')
ax = class_table.plot.bar(x='grade', y='count', legend=False, color='#237a91')
ax.set(title='Original label distribution', xlabel='DR grade', ylabel='Labelled images')
plt.show()


## 4. Match labels to image files

Compare filenames before checking image content. A file can exist without being readable. This step does not modify any images.

In [ ]:
expected_names = {f'{image_id}.png' for image_id in df['id_code']}
actual_names = {path.name for path in IMAGES.glob('*.png')}
missing_names = sorted(expected_names - actual_names)
extra_names = sorted(actual_names - expected_names)
print('CSV rows:', len(df))
print('PNG files:', len(actual_names))
print('Missing:', len(missing_names), missing_names[:5])
print('Extra:', len(extra_names), extra_names[:5])


## 5. Read the full image audit

The audit verifies PNG integrity, fully decodes images, computes file and pixel hashes, records dimensions, and detects exact duplicates and label conflicts. A first run takes several minutes. Existing results are read by default. Set `RERUN_AUDIT = True` after changing the inputs.

Identical pixels can be stored with different compression settings, so both file hashes and pixel hashes are recorded. Flag conflicting labels for review; do not guess a replacement grade.

In [ ]:
RERUN_AUDIT = False
audit_dir = ROOT / 'artifacts/audit'
if RERUN_AUDIT or not (audit_dir / 'audit_summary.json').exists():
    subprocess.run([sys.executable, '-m', 'src.audit_data'], cwd=ROOT, check=True)

audit = json.loads((audit_dir / 'audit_summary.json').read_text(encoding='utf-8'))
if audit['labels_sha256'] != file_hash(LABELS):
    raise RuntimeError('The label file has changed. Rerun the audit.')
inventory = pd.read_csv(audit_dir / 'image_inventory.csv', dtype={'id_code': str})
display(pd.Series({k: v for k, v in audit.items() if isinstance(v, (int, bool))}))
display(inventory[['width', 'height', 'bytes']].describe())


## 6. Inspect local samples and review lists

Sample images illustrate the data appearance; they do not provide a medical diagnosis. The following outputs contain source images. Clear all saved outputs before sharing the Notebook publicly.

The near-duplicate list contains dHash screening candidates only. Similarity does not establish patient identity, and the absence of a candidate does not establish independence.

In [ ]:
display(DisplayImage(filename=str(audit_dir / 'PRIVATE_sample_images.jpg')))
conflicts = pd.read_csv(audit_dir / 'conflicting_labels.csv')
candidates = pd.read_csv(audit_dir / 'near_duplicate_candidates.csv')
print('Conflicting-label images:', len(conflicts))
print('Similar-image candidate pairs:', len(candidates))
display(conflicts.head())
display(candidates.head(10))


## 7. Inspect the reproducible provisional split

The training set is used for learning, the validation set for comparing choices, and the test set is reserved for final evaluation. Each known duplicate group must remain within one set.

The script temporarily excludes records with identical pixels but conflicting grades and aims for 70/15/15 proportions. Group constraints mean these proportions are approximate. Until similar-image and related-patient review is complete, this split cannot establish leakage-free final evaluation.

Confirmed related-image groups can be recorded in local `data/reviewed_groups.csv` using the columns `id_code,reviewed_group`. Use the command in README to generate a reviewed split.

In [ ]:
split_dir = ROOT / 'artifacts/splits'
if not (split_dir / 'split_summary.json').exists():
    subprocess.run([sys.executable, '-m', 'src.split_data'], cwd=ROOT, check=True)
split_summary = json.loads((split_dir / 'split_summary.json').read_text(encoding='utf-8'))
if split_summary['inventory_sha256'] != file_hash(audit_dir / 'image_inventory.csv'):
    raise RuntimeError('The audit has changed. Regenerate the split.')
manifest = pd.read_csv(split_dir / 'split_manifest.csv', dtype={'id_code': str})
display(pd.crosstab(manifest['split'], manifest['diagnosis']))
assert manifest.groupby('group_id')['split'].nunique().max() == 1
assert manifest.groupby('pixel_sha256')['split'].nunique().max() == 1
assert not manifest['id_code'].duplicated().any()
print(split_summary['status'])


## 8. Understand image preprocessing

The baseline converts images to RGB, preserves aspect ratio while resizing, adds black padding to 224 by 224 pixels, and saves separate PNG files. It applies no random augmentation or model-specific normalization. The 224-pixel resolution is an experimental starting point, not evidence that all small lesions are preserved. Black-border cropping is off by default and can be explored separately.

Agree on input size, normalization and training augmentation with the model teammate. Validation and test data must not be randomly augmented or resampled to improve scores.

In [ ]:
processed = ROOT / 'data/processed_224_v1'
if (processed / 'preprocessing_config.json').exists():
    config = json.loads((processed / 'preprocessing_config.json').read_text(encoding='utf-8'))
    display(pd.Series(config))
    display(DisplayImage(filename=str(processed / 'PRIVATE_before_after.jpg')))
else:
    print('Run from project root: python -m src.preprocess')


## 9. Explain the decisions and hand over the data

Be ready to explain these points in your own words:

1. How many images are in each class, and how large is the class imbalance?
2. What is a label conflict, and why should labels not be changed automatically?
3. Why should images not be duplicated or augmented before a random split?
4. Why must identical images not cross training and test boundaries?
5. Can the available CSV support evaluation of elderly patients?
6. What are the benefits and limitations of the 224 by 224 baseline?

Hand over the processing code, instructions, the same reviewed split, preprocessing settings and measured data-quality findings. Obtain authorised dataset access and keep dataset files out of public repositories.

**Outstanding work:** similar-image and label-conflict review, agreement on model inputs, reading primary sources for the disease background, and a complete AI-use appendix. No model has been trained and no clinical performance has been established.

For the B handoff demonstration and file locations, read `B_PRESENTATION_GUIDE.txt`.